In [ ]:
# Use the Kaggle secret when running on Kaggle; locally, authenticate once with `hf auth login`.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login via Kaggle secret: OK')
except ImportError:
    from huggingface_hub import get_token
    if get_token() is None:
        raise RuntimeError(
            'Llama-3.2-1B is gated. Request model access and run `hf auth login` first.'
        )
    print('Using the saved Hugging Face login.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

MODEL_ID = __import__('os').environ.get(
    'ASSIGNMENT_MODEL_PATH', 'meta-llama/Llama-3.2-1B'
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Question 4: q_proj SVD analysis for layers 0, 7, and 15.
from pathlib import Path
import gc
import pandas as pd

OUTPUT_DIR = Path('/kaggle/working/Assignment1') if Path('/kaggle/working').exists() else Path('Assignment1_results')
FIGURE_DIR = OUTPUT_DIR / 'q4_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

svd_device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
energy_rows = []

for layer_idx in (0, 7, 15):
    q_weight = model.model.layers[layer_idx].self_attn.q_proj.weight.detach()
    # Compute in FP32 for a stable spectrum; process one layer at a time.
    _, singular_values, _ = torch.linalg.svd(
        q_weight.to(device=svd_device, dtype=torch.float32),
        full_matrices=False,
    )
    singular_values = singular_values.cpu()
    cumulative_energy = torch.cumsum(singular_values.square(), dim=0)
    cumulative_energy = cumulative_energy / cumulative_energy[-1] * 100.0
    min_rank_50 = int(torch.searchsorted(cumulative_energy, 50.0).item()) + 1
    energy_rows.append({'layer': layer_idx, 'minimum_rank_for_50pct_energy': min_rank_50})

    shown = min(200, singular_values.numel())
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(range(1, shown + 1), singular_values[:shown].numpy(), width=0.85)
    ax.set(title=f'Layer {layer_idx}: q_proj singular values (first {shown})',
           xlabel='Singular value index', ylabel='Magnitude')
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'layer_{layer_idx}_singular_values.png', dpi=180)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(range(1, shown + 1), cumulative_energy[:shown].numpy())
    ax.axhline(50.0, color='red', linestyle='--', linewidth=1, label='50% energy')
    ax.axvline(min_rank_50, color='gray', linestyle=':', linewidth=1,
               label=f'minimum rank = {min_rank_50}')
    ax.set(title=f'Layer {layer_idx}: q_proj cumulative energy (first {shown})',
           xlabel='Target rank', ylabel='Cumulative energy (%)', ylim=(0, 100))
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'layer_{layer_idx}_cumulative_energy.png', dpi=180)
    plt.show()
    plt.close(fig)

    del q_weight, singular_values, cumulative_energy
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

energy_table = pd.DataFrame(energy_rows)
print('Minimum target ranks retaining at least 50% cumulative energy:')
display(energy_table)
energy_table.to_csv(OUTPUT_DIR / 'q4_minimum_ranks.csv', index=False)
print('Saved table and six charts under:', OUTPUT_DIR)
